# GP Classification Tutorial (Binary + OvR)

Purpose: classify safe/failure regions in a 2D load-space with uncertainty around boundaries.


## Learning Roadmap

- Learn binary GP classification uncertainty near decision boundaries.
- Extend to OvR multiclass and inspect entropy maps.
- Check confidence calibration with reliability-style plots.


In [ ]:
# Step 1: import classification models and plotting tools
# Configure Python path for local package imports
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'gp' else Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import math
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)
np.random.seed(42)

from deepuq.models import GaussianProcessClassifier, OneVsRestGaussianProcessClassifier, RBFKernel


In [ ]:
# Step 2: synthesize nonlinear boundary data for binary + multiclass tasks
n = 220
x = torch.empty(n, 2).uniform_(-2.5, 2.5)

# Binary safe/failure region with nonlinear boundary
boundary = 0.8 * torch.sin(1.2 * x[:, 0]) + 0.25 * x[:, 0] - 0.2
y_bin = (x[:, 1] > boundary).float()

# Multiclass bins from risk score
risk = x[:, 1] - boundary
y_multi = torch.bucketize(risk, boundaries=torch.tensor([-0.6, 0.6]))


In [ ]:
# Step 3: fit binary and OvR GP classifiers
binary_gp = GaussianProcessClassifier(
    kernel=RBFKernel(lengthscale=torch.tensor([1.0, 1.0]), outputscale=1.0),
    max_iter=15,
    tol=1e-4,
)
binary_gp.fit(x, y_bin)

ovr_gp = OneVsRestGaussianProcessClassifier(
    kernel=RBFKernel(lengthscale=torch.tensor([1.0, 1.0]), outputscale=1.0),
    max_iter=12,
)
ovr_gp.fit(x, y_multi)


In [ ]:
# Step 4: create prediction grids for uncertainty visualization
grid_x1 = torch.linspace(-2.8, 2.8, 120)
grid_x2 = torch.linspace(-2.8, 2.8, 120)
X1, X2 = torch.meshgrid(grid_x1, grid_x2, indexing='ij')
grid = torch.stack([X1.reshape(-1), X2.reshape(-1)], dim=1)

probs_bin = binary_gp.predict_proba(grid)[:, 1].reshape(120, 120)
probs_multi = ovr_gp.predict_proba(grid)
entropy_multi = -torch.sum(probs_multi * torch.log(probs_multi.clamp_min(1e-8)), dim=1)
entropy_multi = entropy_multi.reshape(120, 120)


In [ ]:
# Step 5: plot binary probability and multiclass entropy maps
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].contourf(X1.numpy(), X2.numpy(), probs_bin.numpy(), levels=20, cmap='viridis')
axes[0].scatter(x[:, 0].numpy(), x[:, 1].numpy(), c=y_bin.numpy(), cmap='coolwarm', s=12, alpha=0.7)
axes[0].set_title('Binary GP: p(failure)')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].contourf(X1.numpy(), X2.numpy(), entropy_multi.numpy(), levels=20, cmap='magma')
axes[1].scatter(x[:, 0].numpy(), x[:, 1].numpy(), c=y_multi.numpy(), cmap='tab10', s=12, alpha=0.7)
axes[1].set_title('OvR multiclass uncertainty (entropy)')
fig.colorbar(im1, ax=axes[1])

for ax in axes:
    ax.set_xlabel('load feature 1')
    ax.set_ylabel('load feature 2')
plt.tight_layout()
plt.show()


In [ ]:
# Additional diagnostic: confidence histogram and reliability-style curve
# This helps understand whether predicted confidence tracks empirical correctness.
train_probs = binary_gp.predict_proba(x)[:, 1]
train_pred = (train_probs >= 0.5).float()
train_correct = (train_pred == y_bin).float()
train_conf = torch.maximum(train_probs, 1 - train_probs)

bins = torch.linspace(0.5, 1.0, 7)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
accs, confs = [], []
for i in range(len(bins) - 1):
    m = (train_conf >= bins[i]) & (train_conf < bins[i + 1])
    if m.any():
        accs.append(train_correct[m].mean().item())
        confs.append(train_conf[m].mean().item())
    else:
        accs.append(float('nan'))
        confs.append(float('nan'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_conf.numpy(), bins=12, alpha=0.8, color='tab:blue')
axes[0].set_title('Binary classifier confidence histogram')
axes[0].set_xlabel('confidence')
axes[0].set_ylabel('count')

axes[1].plot([0.5, 1.0], [0.5, 1.0], 'k--', lw=1)
axes[1].plot(confs, accs, 'o-', color='tab:orange')
axes[1].set_xlim(0.5, 1.0)
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('Reliability-style curve')
axes[1].set_xlabel('mean confidence per bin')
axes[1].set_ylabel('empirical accuracy per bin')
plt.tight_layout()
plt.show()
